In [ ]:
from google.colab import files
files.upload()

Saving filtered_variants_p_1e-08.csv to filtered_variants_p_1e-08.csv
Saving test_dataset_1e-8.txt to test_dataset_1e-8.txt
Saving train_val_dataset_1e-8.txt to train_val_dataset_1e-8.txt


{'filtered_variants_p_1e-08.csv': b'SNP,CHR,BP,P,A1,A2,BETA\r\nrs4575098,1,161155392,1.89639605441e-10,A,G,0.0164116326034849\r\nrs11585858,1,161156033,5.04709041294e-10,A,C,0.0160438710246105\r\nrs4844600,1,207679307,1.1763080598e-14,A,G,0.0214008256640713\r\nrs12037841,1,207684192,2.14484154216e-10,T,G,0.019284895735588\r\nrs4266886,1,207685786,8.16575822488e-16,T,C,0.0223088056509874\r\nrs4562624,1,207685965,1.59536524603e-17,A,C,0.0243820090624837\r\nrs6656401,1,207692049,2.58382931308e-18,A,G,0.0250140516646074\r\nrs6661489,1,207698044,1.64882559696e-16,T,C,0.0228835439379107\r\nrs7515905,1,207738077,6.39983356493e-17,T,C,0.0234115031665153\r\nrs1752684,1,207747296,6.709922986870001e-18,A,G,0.0241508276716563\r\nrs679515,1,207750568,6.83416416409e-19,T,C,0.0254176990093175\r\nrs3818361,1,207784968,1.2772149024800001e-18,A,G,0.0241459827394573\r\nrs6701713,1,207786289,9.82450962732e-19,A,G,0.0242287168739908\r\nrs2093761,1,207786542,1.4693978922500002e-18,A,G,0.0245116022424352\r\n

In [ ]:
# -*- coding: utf-8 -*-

# ══════════════════════════════════════════════════════════════════════════════
# 1. IMPORT
# ══════════════════════════════════════════════════════════════════════════════
import os
import math
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════════
# 2. CONFIG
# ══════════════════════════════════════════════════════════════════════════════

THRESHOLD_LABEL = '1e-8'   # Can be modified to 1e-6 or 1e-4 if needed

FILE_MAP = {
    '1e-8': {
        'train': 'train_val_dataset_1e-8.txt',
        'test' : 'test_dataset_1e-8.txt',
        'meta' : 'filtered_variants_p_1e-08.csv',
    },
    '1e-6': {
        'train': 'train_val_dataset_1e-6.txt',
        'test' : 'test_dataset_1e-6.txt',
        'meta' : 'filtered_variants_p_1e-06.csv',
    },
    '1e-4': {
        'train': 'train_val_dataset_1e-4.txt',
        'test' : 'test_dataset_1e-4.txt',
        'meta' : 'filtered_variants_p_0.0001.csv',
    },
}

CONFIG = {
    'threshold_label' : THRESHOLD_LABEL,
    'train_path'      : FILE_MAP[THRESHOLD_LABEL]['train'],
    'test_path'       : FILE_MAP[THRESHOLD_LABEL]['test'],
    'meta_path'       : FILE_MAP[THRESHOLD_LABEL]['meta'],
    'output_dir'      : 'results_grafaid',

    # Grid search
    'k_values'        : [2,3,4,5,6,7,8],
    'dropout_values'  : [0.1, 0.2, 0.3, 0.4, 0.5],

    'n_folds'         : 5,
    'epochs'          : 300,
    'cv_epochs'       : 300,
    'lr'              : 1e-4,
    'weight_decay'    : 1e-4,
    'patience'        : 40,
    'seed'            : 42,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PALETTE = {
    'primary'  : '#2563EB',
    'secondary': '#DC2626',
    'accent'   : '#16A34A',
    'neutral'  : '#6B7280',
    'bg'       : '#F8FAFC',
}


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG['seed'])
print(f"Thiết bị : {DEVICE} | PyTorch: {torch.__version__}")
print(f"Ngưỡng   : p < {CONFIG['threshold_label']}")
os.makedirs(CONFIG['output_dir'], exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════════
# Data Preprocessing
# ══════════════════════════════════════════════════════════════════════════════

def load_data(train_path, test_path, meta_path):
    train_df = pd.read_csv(train_path, sep=' ')
    test_df  = pd.read_csv(test_path,  sep=' ')
    snp_cols = [c for c in train_df.columns if c not in ('RID', 'Phenotype')]

    X_train    = train_df[snp_cols].values.astype(np.float32)
    y_train    = train_df['Phenotype'].values.astype(np.int64)
    rids_train = train_df['RID'].values

    X_test     = test_df[snp_cols].values.astype(np.float32)
    y_test     = test_df['Phenotype'].values.astype(np.int64)
    rids_test  = test_df['RID'].values

    meta_df = pd.read_csv(meta_path)

    print(f"\n{'─'*55}")
    print(f"  Train : {X_train.shape} | CN={int((y_train==0).sum())} AD={int((y_train==1).sum())}")
    print(f"  Test  : {X_test.shape}  | CN={int((y_test==0).sum())} AD={int((y_test==1).sum())}")
    print(f"  SNPs  : {len(snp_cols)}")
    print(f"  hidden_dim sẽ dùng: int(√{len(snp_cols)}) = {int(math.sqrt(len(snp_cols)))}")
    print(f"{'─'*55}\n")

    return (X_train, y_train, rids_train,
            X_test,  y_test,  rids_test,
            snp_cols, meta_df)



def cosine_distance_torch(X: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    w = X.norm(p=2, dim=1, keepdim=True)
    return 1.0 - torch.mm(X, X.t()) / (w * w.t()).clamp(min=eps)


def cal_adj_parameter(edge_per_node: int,
                      data: torch.Tensor) -> float:
    dist = cosine_distance_torch(data)
    N    = data.shape[0]
    parameter = torch.sort(dist.reshape(-1)).values[edge_per_node * N]
    return parameter.item()


def graph_from_dist(dist: torch.Tensor,
                    parameter: float,
                    self_dist: bool = True) -> torch.Tensor:
    g = (dist <= parameter).float()
    if self_dist:
        idx = np.diag_indices(g.shape[0])
        g[idx[0], idx[1]] = 0.0
    return g


def build_transductive_adj(X_train: np.ndarray,
                            X_test:  np.ndarray,
                            edge_per_node: int) -> tuple:
    N_tr = X_train.shape[0]
    N_te = X_test.shape[0]
    N    = N_tr + N_te

    # Gộp train + test thành một tensor
    data_all = torch.FloatTensor(
        np.vstack([X_train, X_test])
    ).to(DEVICE)

    tr_idx = list(range(N_tr))
    te_idx = list(range(N_tr, N))

    # Tính parameter ngưỡng từ train data
    data_tr = data_all[tr_idx]
    parameter = cal_adj_parameter(edge_per_node, data_tr)

    # ── Xây ma trận kề đầy đủ (N, N) ────────────────────────────────────────
    adj = torch.zeros(N, N, device=DEVICE)

    # Block tr-tr
    dist_tr2tr = cosine_distance_torch(data_all[tr_idx])
    g_tr2tr    = graph_from_dist(dist_tr2tr, parameter, self_dist=True)
    adj[:N_tr, :N_tr] = (1 - dist_tr2tr) * g_tr2tr

    # Block tr-te
    dist_tr2te = cosine_distance_torch(
        data_all[tr_idx]
    ) if False else _cross_dist(data_all[tr_idx], data_all[te_idx])
    g_tr2te    = graph_from_dist(dist_tr2te, parameter, self_dist=False)
    adj[:N_tr, N_tr:] = (1 - dist_tr2te) * g_tr2te

    # Block te-tr
    dist_te2tr = _cross_dist(data_all[te_idx], data_all[tr_idx])
    g_te2tr    = graph_from_dist(dist_te2tr, parameter, self_dist=False)
    adj[N_tr:, :N_tr] = (1 - dist_te2tr) * g_te2tr

    # Đảm bảo đối xứng (giữ giá trị lớn hơn)
    adj_T = adj.t()
    adj   = adj + adj_T * (adj_T > adj).float() - adj * (adj_T > adj).float()

    I   = torch.eye(N, device=DEVICE)
    adj = F.normalize(adj + I, p=1, dim=1)

    # Chuyển sang sparse để tiết kiệm bộ nhớ
    adj_sparse = _to_sparse(adj)

    return adj_sparse, tr_idx, te_idx


def _cross_dist(X_a: torch.Tensor,
                X_b: torch.Tensor,
                eps: float = 1e-8) -> torch.Tensor:
    w_a = X_a.norm(p=2, dim=1, keepdim=True)
    w_b = X_b.norm(p=2, dim=1, keepdim=True)
    sim = torch.mm(X_a, X_b.t()) / (w_a * w_b.t()).clamp(min=eps)
    return 1.0 - sim


def _to_sparse(x: torch.Tensor) -> torch.Tensor:
    indices = torch.nonzero(x, as_tuple=False)
    if indices.shape[0] == 0:
        return x.to_sparse()
    values = x[indices[:, 0], indices[:, 1]]
    return torch.sparse_coo_tensor(
        indices.t(), values, x.size(), device=x.device
    )


# ══════════════════════════════════════════════════════════════════════════════
# KIẾN TRÚC GCN
# ══════════════════════════════════════════════════════════════════════════════

class UniformDropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        self.beta = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            noise = (torch.rand_like(x) * self.beta * 2) - self.beta
            return x * (1 + noise)
        return x


class GraphConvolution(nn.Module):
    def __init__(self, in_features: int, out_features: int,
                 dropout: float, bias: bool = True):
        super().__init__()
        self.weight   = nn.Parameter(
            torch.FloatTensor(in_features, out_features))
        self.bias_vec = nn.Parameter(
            torch.FloatTensor(out_features)) if bias else None
        self.dropout  = UniformDropout(dropout)
        self.act      = nn.SELU()

        nn.init.xavier_normal_(self.weight, gain=3/4)
        if self.bias_vec is not None:
            self.bias_vec.data.fill_(0.0)

    def forward(self, x: torch.Tensor,
                adj: torch.Tensor) -> torch.Tensor:
        support = torch.mm(x, self.weight)
        output  = torch.sparse.mm(adj, support)
        output  = self.dropout(output)
        output  = self.act(output)
        if self.bias_vec is not None:
            output = output + self.bias_vec
        return output


class GCN(nn.Module):
    def __init__(self, in_dim: int, dropout: float):
        super().__init__()
        hidden_dim = int(math.sqrt(in_dim))   # quy tắc √D của paper gốc

        self.gc1 = GraphConvolution(in_dim,     hidden_dim, dropout)
        self.gc2 = GraphConvolution(hidden_dim, hidden_dim, dropout)

        # Classifier: 2-class (CN=0, AD=1)
        self.clf = nn.Linear(hidden_dim, 2)
        nn.init.xavier_normal_(self.clf.weight, gain=1.0)
        self.clf.bias.data.fill_(0.0)

        self.hidden_dim = hidden_dim

    def forward(self, x: torch.Tensor,
                adj: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x   : (N, D) — feature matrix toàn bộ nodes (train + test)
            adj : (N, N) sparse — adjacency matrix

        Returns:
            logit: (N, 2) — raw logit trước softmax
        """
        x = self.gc1(x, adj)   # (N, hidden_dim)
        x = self.gc2(x, adj)   # (N, hidden_dim)
        return self.clf(x)     # (N, 2)


# ══════════════════════════════════════════════════════════════════════════════
# 6. TRAINING
# ══════════════════════════════════════════════════════════════════════════════

def train_one_fold(X_fold_train: np.ndarray,
                   y_fold_train: np.ndarray,
                   X_fold_val:   np.ndarray,
                   y_fold_val:   np.ndarray,
                   edge_per_node: int,
                   dropout:       float,
                   max_epochs:    int,
                   lr:            float,
                   weight_decay:  float,
                   patience:      int) -> float:
    D = X_fold_train.shape[1]

    # Xây adjacency transductive cho fold
    adj_fold, tr_idx, te_idx = build_transductive_adj(
        X_fold_train, X_fold_val, edge_per_node)

    # Data tensor toàn bộ fold (train + val)
    X_all = torch.FloatTensor(
        np.vstack([X_fold_train, X_fold_val])
    ).to(DEVICE)
    adj_fold = adj_fold.to(DEVICE)

    # Nhãn — chỉ cần train nhãn để tính loss
    y_tr_t = torch.LongTensor(y_fold_train).to(DEVICE)

    # Class weight: cc_ratio = n_control / n_case (theo paper gốc)
    n_case    = int((y_fold_train == 1).sum())
    n_control = int((y_fold_train == 0).sum())
    cc_ratio  = n_control / max(n_case, 1)
    weights   = torch.tensor([1.0, cc_ratio], dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    set_seed(CONFIG['seed'])
    model     = GCN(D, dropout).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_epochs)

    best_val_auc   = 0.0
    best_state     = None
    patience_count = 0

    for epoch in range(max_epochs):
        # ── Training step ──────────────────────────────────────────────────
        model.train()
        optimizer.zero_grad()

        logit_all = model(X_all, adj_fold)           # (N_all, 2)
        logit_tr  = logit_all[tr_idx]                # (N_tr, 2)
        loss      = criterion(logit_tr, y_tr_t)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        # ── Validation mỗi 5 epoch ────────────────────────────────────────
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                logit_all = model(X_all, adj_fold)
                prob_te   = F.softmax(logit_all[te_idx], dim=1)
                prob_ad   = prob_te[:, 1].cpu().numpy()   # xác suất lớp AD

            try:
                val_auc = roc_auc_score(y_fold_val, prob_ad)
            except Exception:
                val_auc = 0.0

            if val_auc > best_val_auc:
                best_val_auc   = val_auc
                best_state     = {k: v.cpu().clone()
                                  for k, v in model.state_dict().items()}
                patience_count = 0
            else:
                patience_count += 1

            if patience_count >= patience // 5:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val_auc


# ══════════════════════════════════════════════════════════════════════════════
# 7. GRID SEARCH
# ══════════════════════════════════════════════════════════════════════════════

def grid_search_cv(X_train: np.ndarray,
                   y_train: np.ndarray,
                   config:  dict) -> dict:

    skf    = StratifiedKFold(n_splits=config['n_folds'],
                             shuffle=True, random_state=config['seed'])
    D      = X_train.shape[1]
    hid    = int(math.sqrt(D))

    k_vals   = config['k_values']
    drp_vals = config['dropout_values']
    total    = len(k_vals) * len(drp_vals)
    run      = 0

    print(f"\nGrid search: {total} tổ hợp × {config['n_folds']} folds")
    print(f"D={D} → hidden_dim=√{D}={hid} (cố định theo paper gốc)")
    print(f"k ∈ {k_vals} (edge_per_node), dropout ∈ {drp_vals}\n")

    best_auc    = -1.0
    best_params = {}
    results_log = []

    for k in k_vals:
        for dropout in drp_vals:
            run += 1
            fold_aucs = []
            t0 = time.time()

            for tr_idx, val_idx in skf.split(X_train, y_train):
                val_auc = train_one_fold(
                    X_train[tr_idx], y_train[tr_idx],
                    X_train[val_idx], y_train[val_idx],
                    edge_per_node=k,
                    dropout=dropout,
                    max_epochs=config['cv_epochs'],
                    lr=config['lr'],
                    weight_decay=config['weight_decay'],
                    patience=config['patience'],
                )
                fold_aucs.append(val_auc)

            m, s = float(np.mean(fold_aucs)), float(np.std(fold_aucs))
            results_log.append({
                'k': k, 'hidden_dim': hid, 'dropout': dropout,
                'mean_val_auc': m, 'std_val_auc': s,
            })
            print(f"  [{run:2d}/{total}] k={k:2d} | "
                  f"hidden={hid:3d} | dropout={dropout:.1f} | "
                  f"val AUC={m:.4f}±{s:.4f}  ({time.time()-t0:.0f}s)")

            if m > best_auc:
                best_auc    = m
                best_params = {'k': k, 'hidden_dim': hid, 'dropout': dropout}

    print(f"\n[INFO] Best: {best_params} | val AUC={best_auc:.4f}")

    label  = config['threshold_label']
    log_df = (pd.DataFrame(results_log)
              .sort_values('mean_val_auc', ascending=False))
    log_df.to_csv(
        os.path.join(config['output_dir'], f'grid_search_log_{label}.csv'),
        index=False)
    print(f"[INFO] Grid search log saved.")
    return best_params


# ══════════════════════════════════════════════════════════════════════════════
# 8. FINAL MODEL + EVALUATE
# ══════════════════════════════════════════════════════════════════════════════

def train_final_and_evaluate(X_train:    np.ndarray,
                              y_train:    np.ndarray,
                              X_test:     np.ndarray,
                              y_test:     np.ndarray,
                              rids_test:  np.ndarray,
                              best_params: dict,
                              config:     dict):
    k       = best_params['k']
    dropout = best_params['dropout']
    D       = X_train.shape[1]
    label   = config['threshold_label']

    print(f"\n[INFO] Final training: k={k}, hidden={int(math.sqrt(D))}, "
          f"dropout={dropout}")

    t0 = time.time()
    adj_final, tr_idx, te_idx = build_transductive_adj(
        X_train, X_test, edge_per_node=k)
    print(f"[INFO] Xong trong {time.time()-t0:.1f}s")

    # Data tensor
    X_all   = torch.FloatTensor(
        np.vstack([X_train, X_test])
    ).to(DEVICE)
    adj_final = adj_final.to(DEVICE)
    y_tr_t  = torch.LongTensor(y_train).to(DEVICE)

    # Class weight
    n_case    = int((y_train == 1).sum())
    n_control = int((y_train == 0).sum())
    cc_ratio  = n_control / max(n_case, 1)
    weights   = torch.tensor([1.0, cc_ratio], dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    set_seed(config['seed'])
    model     = GCN(D, dropout).to(DEVICE)
    optimizer = optim.Adam(model.parameters(),
                           lr=config['lr'],
                           weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['epochs'])

    losses = []
    print(f"[INFO] Training {config['epochs']} epochs...")

    for epoch in range(config['epochs']):
        model.train()
        optimizer.zero_grad()

        logit_all = model(X_all, adj_final)
        logit_tr  = logit_all[tr_idx]
        loss      = criterion(logit_tr, y_tr_t)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1:3d}/{config['epochs']} | "
                  f"loss={loss.item():.4f}")

    os.makedirs('models', exist_ok=True)
    torch.save(model.state_dict(),
               f"models/gcn_grafaid_{label}.pt")
    print(f"[INFO] Model saved: models/gcn_grafaid_{label}.pt")

    # ── Inference tại test nodes ──────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        logit_all = model(X_all, adj_final)
        prob_all  = F.softmax(logit_all, dim=1)
        y_prob    = prob_all[te_idx, 1].cpu().numpy()   # xác suất AD

    y_pred = (y_prob >= 0.5).astype(int)

    # ── Metrics ───────────────────────────────────────────────────────────────
    metrics = {
        'Accuracy'  : float(accuracy_score(y_test, y_pred)),
        'Precision' : float(precision_score(y_test, y_pred, zero_division=0)),
        'Recall'    : float(recall_score(y_test, y_pred, zero_division=0)),
        'F1-Score'  : float(f1_score(y_test, y_pred, zero_division=0)),
        'AUC-ROC'   : float(roc_auc_score(y_test, y_prob)),
        'AUPRC'     : float(average_precision_score(y_test, y_prob)),
    }

    print(f"\n{'─'*50}")
    print(f"  KẾT QUẢ TEST — p < {label}")
    print(f"{'─'*50}")
    for name, val in metrics.items():
        print(f"  {name:<12}: {val:.4f}")
    print(f"{'─'*50}")

    # Save to CSV
    prob_df = pd.DataFrame({
        'RID'           : rids_test,
        'True_Label'    : y_test.astype(int),
        'True_Diagnosis': np.where(y_test == 1, 'AD', 'CN'),
        'Pred_Label'    : y_pred,
        'Pred_Diagnosis': np.where(y_pred == 1, 'AD', 'CN'),
        'Pred_Prob'     : np.round(y_prob, 6),
        'Correct'       : (y_test.astype(int) == y_pred).astype(int),
    }).sort_values('Pred_Prob', ascending=False).reset_index(drop=True)

    csv_path = os.path.join(config['output_dir'],
                            f'test_probabilities_grafaid_{label}.csv')
    prob_df.to_csv(csv_path, index=False)
    print(f"[INFO] Saved: {csv_path}")

    # ── Lưu metrics ──────────────────────────────────────────────────────────
    pd.DataFrame([{
        'threshold': label, **best_params, **metrics,
    }]).to_csv(
        os.path.join(config['output_dir'], f'metrics_grafaid_{label}.csv'),
        index=False)
    print(f"[INFO] Saved: metrics_grafaid_{label}.csv")

    return metrics, y_test, y_prob, y_pred, losses


# ══════════════════════════════════════════════════════════════════════════════
# 9. VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

def plot_confusion_matrix(y_true, y_pred, label, output_dir):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fig, ax = plt.subplots(figsize=(5, 4))
    fig.patch.set_facecolor(PALETTE['bg'])
    ax.set_facecolor(PALETTE['bg'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['CN (0)', 'AD (1)'],
                yticklabels=['CN (0)', 'AD (1)'],
                linewidths=0.5, linecolor='white',
                annot_kws={'size': 14, 'weight': 'bold'},
                ax=ax, cbar=True)
    ax.set_title(f'Confusion Matrix — GRAF-AID (p < {label})',
                 fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel(f'Predicted Label\nTN={tn}  FP={fp}  FN={fn}  TP={tp}',
                  fontsize=9)
    ax.set_ylabel('True Label', fontsize=10)
    plt.tight_layout()
    path = os.path.join(output_dir, f'confusion_matrix_grafaid_{label}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Saved: {path}")


def plot_roc_curve(y_true, y_prob, label, output_dir):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 5))
    fig.patch.set_facecolor(PALETTE['bg'])
    ax.set_facecolor(PALETTE['bg'])
    ax.plot(fpr, tpr, color=PALETTE['primary'], lw=2,
            label=f'GRAF-AID (AUC = {auc_val:.4f})')
    ax.plot([0,1],[0,1], color=PALETTE['neutral'], lw=1,
            linestyle='--', label='Random classifier')
    ax.fill_between(fpr, tpr, alpha=0.08, color=PALETTE['primary'])
    ax.set(xlim=[0,1], ylim=[0,1.05],
           xlabel='False Positive Rate', ylabel='True Positive Rate')
    ax.set_title(f'ROC Curve — GRAF-AID (p < {label})',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(output_dir, f'roc_curve_grafaid_{label}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Saved: {path}")


def plot_risk_distribution(y_true, y_prob, label, output_dir):
    fig, ax = plt.subplots(figsize=(6, 4))
    fig.patch.set_facecolor(PALETTE['bg'])
    ax.set_facecolor(PALETTE['bg'])
    bins = np.linspace(0, 1, 30)
    ax.hist(y_prob[y_true==0], bins=bins, alpha=0.65,
            color=PALETTE['accent'],
            label=f"CN (n={(y_true==0).sum()})",
            edgecolor='white', linewidth=0.4)
    ax.hist(y_prob[y_true==1], bins=bins, alpha=0.65,
            color=PALETTE['secondary'],
            label=f"AD (n={(y_true==1).sum()})",
            edgecolor='white', linewidth=0.4)
    ax.axvline(0.5, color=PALETTE['neutral'],
               linestyle='--', lw=1.5, label='Threshold = 0.5')
    ax.set(xlabel='Predicted Risk Score (Probability)', ylabel='Count')
    ax.set_title(f'Risk Score Distribution — GRAF-AID (p < {label})',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(output_dir,
                        f'risk_score_distribution_grafaid_{label}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Saved: {path}")


def plot_pr_curve(y_true, y_prob, label, output_dir):
    prec_v, rec_v, _ = precision_recall_curve(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 5))
    fig.patch.set_facecolor(PALETTE['bg'])
    ax.set_facecolor(PALETTE['bg'])
    ax.plot(rec_v, prec_v, color=PALETTE['accent'], lw=2,
            label=f'GRAF-AID (AUPRC = {auprc:.4f})')
    ax.axhline(y_true.mean(), color=PALETTE['neutral'], lw=1,
               linestyle='--', label=f'Baseline = {y_true.mean():.2f}')
    ax.fill_between(rec_v, prec_v, alpha=0.08, color=PALETTE['accent'])
    ax.set(xlim=[0,1], ylim=[0,1.05],
           xlabel='Recall', ylabel='Precision')
    ax.set_title(f'Precision-Recall Curve — GRAF-AID (p < {label})',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(output_dir, f'pr_curve_grafaid_{label}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Saved: {path}")


def plot_training_loss(losses, label, output_dir):
    fig, ax = plt.subplots(figsize=(7, 4))
    fig.patch.set_facecolor(PALETTE['bg'])
    ax.set_facecolor(PALETTE['bg'])
    ax.plot(range(1, len(losses)+1), losses,
            color=PALETTE['primary'], lw=1.5)
    ax.set(xlabel='Epoch', ylabel='CrossEntropy Loss (class-weighted)',
           title=f'Training Loss — GRAF-AID (p < {label})')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(output_dir, f'training_loss_grafaid_{label}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Saved: {path}")


# ══════════════════════════════════════════════════════════════════════════════
# 10. MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    label = CONFIG['threshold_label']
    print("=" * 60)
    print(f"  GRAF-AID (updated) — p < {label}")
    print(f"  Thiết bị: {DEVICE}")
    print("=" * 60)

    print("\n[STEP 1] Đọc dữ liệu...")
    (X_train, y_train, rids_train,
     X_test,  y_test,  rids_test,
     snp_names, meta_df) = load_data(
        CONFIG['train_path'],
        CONFIG['test_path'],
        CONFIG['meta_path'],
    )

    print(f"\n[STEP 2] Grid Search + {CONFIG['n_folds']}-Fold CV...")
    best_params = grid_search_cv(X_train, y_train, CONFIG)

    print(f"\n[STEP 3] Final model + Evaluate...")
    metrics, y_true, y_prob, y_pred, losses = train_final_and_evaluate(
        X_train, y_train,
        X_test,  y_test, rids_test,
        best_params, CONFIG,
    )

    print(f"\n[STEP 4] Vẽ biểu đồ...")
    plot_confusion_matrix(y_true, y_pred,  label, CONFIG['output_dir'])
    plot_roc_curve(y_true, y_prob,         label, CONFIG['output_dir'])
    plot_risk_distribution(y_true, y_prob, label, CONFIG['output_dir'])
    plot_pr_curve(y_true, y_prob,          label, CONFIG['output_dir'])
    plot_training_loss(losses,             label, CONFIG['output_dir'])

    print(f"\n{'═'*60}")
    print(f"  HOÀN THÀNH — GRAF-AID (p < {label})")
    print(f"{'═'*60}")
    print(f"  Best params : {best_params}")
    for name, val in metrics.items():
        print(f"  {name:<12}: {val:.4f}")
    print(f"{'═'*60}")
    print(f"\n  Output dir: {CONFIG['output_dir']}/")

    return metrics, best_params


if __name__ == "__main__":
    metrics, best_params = main()

Thiết bị : cuda | PyTorch: 2.10.0+cu128
Ngưỡng   : p < 1e-8
  GRAF-AID (updated) — p < 1e-8
  Thiết bị: cuda

[STEP 1] Đọc dữ liệu...

───────────────────────────────────────────────────────
  Train : (1000, 1724) | CN=486 AD=514
  Test  : (174, 1724)  | CN=84 AD=90
  SNPs  : 1724
  hidden_dim sẽ dùng: int(√1724) = 41
───────────────────────────────────────────────────────


[STEP 2] Grid Search + 5-Fold CV...

Grid search: 35 tổ hợp × 5 folds
D=1724 → hidden_dim=√1724=41 (cố định theo paper gốc)
k ∈ [2, 3, 4, 5, 6, 7, 8] (edge_per_node), dropout ∈ [0.1, 0.2, 0.3, 0.4, 0.5]

  [ 1/35] k= 2 | hidden= 41 | dropout=0.1 | val AUC=0.6903±0.0071  (3s)
  [ 2/35] k= 2 | hidden= 41 | dropout=0.2 | val AUC=0.6910±0.0084  (3s)
  [ 3/35] k= 2 | hidden= 41 | dropout=0.3 | val AUC=0.6937±0.0085  (3s)
  [ 4/35] k= 2 | hidden= 41 | dropout=0.4 | val AUC=0.6956±0.0099  (3s)
  [ 5/35] k= 2 | hidden= 41 | dropout=0.5 | val AUC=0.6966±0.0111  (3s)
  [ 6/35] k= 3 | hidden= 41 | dropout=0.1 | val AUC=0.6834